# Unit 5 / Chapter 5: Quantum AI in the Real World

> **Main Learning Objective:** See where quantum AI is being applied today, understand the limits of current hardware, think through the ethics, and design your own industry proposal.

| Section | Topic |
|---|---|
| 5.1 | Quantum AI in industry: finance, drug discovery, materials |
| 5.2 | Hardware reality check: NISQ and where we actually are |
| 5.3 | Ethical implications and workforce impact |
| 5.4 | Final project: your own industry proposal |

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite, Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
# Standard imports for the unit.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(5); random.seed(5)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 5**. You will be asked to enter the email you signed up with so we can track your progress.

In [ ]:
# ============================================================
# COURSE TRACKING - do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 5
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 5.1: Quantum AI in Industry

Quantum methods are already being explored across many industries. Three of the most active ones are **finance**, **drug discovery and medicine**, and **materials discovery**. In each case, the underlying problem is essentially "search a giant space for the best option," which is exactly what quantum optimization and sampling are good at.

## Finance

Banks, hedge funds, and trading firms have been some of the earliest adopters. The most studied applications are:

| Use case | What is being optimized | Quantum tool |
|---|---|---|
| **Portfolio optimization** | Pick a mix of assets that maximizes return for a given risk | QAOA, VQE on a QUBO |
| **Option pricing** | Compute the fair value of a financial derivative | Quantum Monte Carlo, amplitude estimation |
| **Fraud detection** | Spot suspicious transaction patterns | Quantum kernels, quantum ML classifiers |
| **Credit risk scoring** | Estimate likelihood a borrower defaults | Quantum-enhanced sampling, Bayesian methods |

Goldman Sachs, JPMorgan, and HSBC have all published papers showing prototypes on quantum hardware. None of these beat classical computers in production yet, but the research is real and well-funded.

## Drug discovery and medicine

The promise here is unusually strong because biological molecules are quantum systems by nature. Today's high-impact research areas:

| Use case | What is being optimized |
|---|---|
| **Molecular simulation** | Predict molecule properties (energy, reactivity) without lab work |
| **Protein folding** | Find the 3D shape a protein takes, useful for drug design |
| **Drug-target binding** | Score how well a candidate drug attaches to a target receptor |
| **Genomic data analysis** | Find patterns in DNA sequences using quantum ML |

VQE was *invented* partly for chemistry: finding the ground-state energy of a small molecule's Hamiltonian. Companies like Roche, Boehringer Ingelheim, Cleveland Clinic, and several biotech startups have public collaborations with quantum-hardware vendors.

## Materials discovery

The same chemistry techniques that help with drugs also help with materials:

| Use case | Examples |
|---|---|
| **Battery materials** | Faster-charging, longer-lasting lithium and post-lithium chemistries |
| **Catalysts** | Industrial catalysts for fertilizer (ammonia), carbon capture |
| **Superconductors** | Materials that conduct electricity with no loss, including at higher temperatures |
| **Photovoltaics** | More efficient solar cell materials |
| **Magnets** | Stronger, rare-earth-free magnets for motors |

A common thread: the cost function in all three industries can be written as "find the lowest-energy configuration of an enormous space," and that's exactly the problem VQE and QAOA are designed for.

### Visualization: estimated industry impact

Researchers and analysts have made (very rough) estimates of where the largest economic impact from quantum AI might land first. The chart below shows a relative-impact picture based on public industry reports through ~2025. Take the absolute numbers with a grain of salt; the comparison shape is what's interesting.

In [ ]:
# Approximate relative-impact estimates (no actual money figures, just relative scale).
industries = ["Finance", "Drug discovery\n& medicine", "Materials\ndiscovery",
              "Logistics", "Energy", "Cybersecurity", "Climate /\nclimate models"]
relative = [9, 10, 8, 6, 7, 5, 4]      # 1-10 relative score, rough public-report consensus

colors = ['#5B2C91', '#2A9D8F', '#E76F51', '#bdbdbd', '#bdbdbd', '#bdbdbd', '#bdbdbd']
order = np.argsort(relative)[::-1]
industries_sorted = [industries[i] for i in order]
relative_sorted   = [relative[i]   for i in order]
colors_sorted     = [colors[i]     for i in order]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(industries_sorted, relative_sorted, color=colors_sorted)
ax.set_ylabel("Relative quantum AI impact score (rough)")
ax.set_ylim(0, 12)
ax.set_title("Where quantum AI is being applied first (the top three colored)")
for b, v in zip(bars, relative_sorted):
    ax.text(b.get_x()+b.get_width()/2, v+0.2, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()
print("The three colored bars are the focus of this section: finance, drug discovery, materials.")

### Activity 5.1: Case study

Pick **one** of the three industries above and answer these prompts in your head or on paper. We will collect short written answers in the end-of-unit form.

1. **What classical method is currently used?** (For example, in portfolio optimization the classical baseline is mean-variance optimization with a quadratic solver.)
2. **Where does that method struggle?** (E.g., too slow when there are too many assets; gets stuck in local minima.)
3. **Which quantum tool from Units 2 through 4 would you try?** (Grover, QFT, VQE, QAOA, quantum kernel, etc.) Why?
4. **What would success look like?** A speedup? A better answer? A new capability?

The code cell below is just a notepad you can type into; nothing happens to it automatically, but you will paste these answers into the end-of-unit quiz at the bottom of the notebook.

In [ ]:
# ---- Your case study scratchpad ----
my_industry = "(finance, drug discovery, or materials)"
classical_method  = "..."
where_it_struggles = "..."
quantum_tool       = "(VQE / QAOA / Grover / quantum kernel / ...)"
why_that_tool      = "..."
success_looks_like = "..."

print("Industry:", my_industry)
print("Classical method:", classical_method)
print("Where it struggles:", where_it_struggles)
print("Quantum tool I would try:", quantum_tool, "because", why_that_tool)
print("Success would look like:", success_looks_like)

---
# Section 5.2: Hardware Reality Check

## NISQ: where we actually are

We are in the **NISQ era**: Noisy Intermediate-Scale Quantum. Coined by physicist John Preskill in 2018, the term captures three honest constraints of today's hardware:

- **Noisy.** Gates have errors of around 0.1% to 1%. After a few hundred gates, those errors compound and the signal is gone.
- **Intermediate-scale.** Devices have roughly 50 to a few thousand physical qubits, far short of the millions needed for fault-tolerant algorithms like Shor's.
- **Quantum.** They genuinely exploit superposition and entanglement, so they are not classical simulators.

The practical consequence: **shallow circuits with few qubits** are realistic; **deep circuits with many qubits** are not. That's why hybrid algorithms like VQE and QAOA dominate the current landscape. They keep the quantum circuit short and lean on classical optimization.

## Current hardware types

There are several competing physical platforms. Each has trade-offs.

| Platform | How it works | Strengths | Weaknesses | Notable companies |
|---|---|---|---|---|
| **Superconducting qubits** | Loops of supercurrent at near-absolute-zero | Fast gates, mature fabrication | Needs dilution refrigerators, short coherence | IBM, Google, Rigetti, IQM |
| **Trapped ions** | Individual atoms held by lasers in vacuum | Very low error rates, all-to-all connectivity | Slower gates, harder to scale | IonQ, Quantinuum, AQT |
| **Neutral atoms** | Atoms held in optical tweezers | Naturally scalable, programmable connectivity | Younger tech, gate fidelities improving | QuEra, Pasqal, Atom Computing |
| **Photonic** | Single photons routed through circuits | Room temperature, networkable | Hard to make two-qubit gates; probabilistic | PsiQuantum, Xanadu, ORCA |
| **Topological** | Exotic states of matter (Majorana) | Theoretically error-resistant | Still experimental, no working qubit yet at scale | Microsoft Azure Quantum |

No platform has clearly won. As of 2025, IBM and Google lead in superconducting; IonQ and Quantinuum lead in trapped ion fidelity; QuEra and Atom Computing lead in neutral-atom scale; PsiQuantum is the photonic frontrunner; Microsoft is betting big on topological.

### Visualization: qubit count vs error rate

A useful rough mental model: more qubits or lower error rate moves you toward useful quantum advantage. The plot below shows approximate **public** numbers from the major platforms in early 2025. Top-right (many qubits, very low error) is the goal. Most platforms still cluster in the messy middle.

In [ ]:
# Approximate public 2025 numbers. Real values shift constantly; this is just a sketch.
platforms = [
    {"name":"IBM (superconducting)",   "qubits": 1121, "err": 1.0e-2, "color":"#5B2C91"},
    {"name":"Google (superconducting)", "qubits": 105,  "err": 5.0e-3, "color":"#5B2C91"},
    {"name":"IonQ (trapped ion)",      "qubits": 36,   "err": 1.0e-3, "color":"#2A9D8F"},
    {"name":"Quantinuum (trapped ion)", "qubits": 56,   "err": 5.0e-4, "color":"#2A9D8F"},
    {"name":"QuEra (neutral atom)",    "qubits": 256,  "err": 5.0e-3, "color":"#E76F51"},
    {"name":"Atom Computing (neutral)", "qubits": 1180, "err": 3.0e-3, "color":"#E76F51"},
    {"name":"Xanadu (photonic)",       "qubits": 216,  "err": 2.0e-2, "color":"#f1c40f"},
]

fig, ax = plt.subplots(figsize=(9, 5.5))
for p in platforms:
    ax.scatter(p["qubits"], p["err"], s=110, color=p["color"], edgecolor="black", zorder=3)
    ax.annotate(p["name"], (p["qubits"], p["err"]),
                xytext=(7,5), textcoords="offset points", fontsize=8.5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xlabel("Physical qubits (log scale)")
ax.set_ylabel("Two-qubit gate error rate (log scale, lower is better)")
ax.set_title("Where major platforms stood around 2025 (approximate public numbers)")
ax.axhspan(1e-5, 1e-4, color='#d8f3ef', alpha=0.5, label="fault-tolerant target (~1e-4 or better)")
ax.axvspan(1e6, 1e8, color='#fde7e0', alpha=0.5, label="scale needed for useful algorithms")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower left")
plt.tight_layout(); plt.show()
print("Reality check: no platform sits in BOTH the green-error band AND the pink-scale band yet.")
print("That gap is why we are in the NISQ era and why hybrid algorithms dominate today.")

### Activity 5.2: Match the algorithm to the hardware

For each of the following AI tasks, which hardware platform would you tentatively pick today, and why? Fill in your reasoning below. Pick from: **superconducting**, **trapped ion**, **neutral atom**, **photonic**.

1. A short variational circuit (10 to 20 qubits, depth 10) for a small portfolio optimization.
2. A very precise small-molecule energy calculation where every gate must be near-perfect.
3. A massive optimization problem over hundreds of variables where error per gate matters less than raw qubit count.
4. A networked sensing application where qubits need to be transmitted between cities.

There are no single right answers; the goal is to reason from the trade-offs in the table above.

In [ ]:
# ---- Activity 5.2 ----
my_picks = {
    "1_short_variational_optimization":   "...",
    "2_high_precision_chemistry":         "...",
    "3_large_combinatorial_optimization": "...",
    "4_networked_sensing":                "...",
}
for k, v in my_picks.items():
    print(k, "->", v)

# Reasoning notes (free-form):
notes = '''
1) ...
2) ...
3) ...
4) ...
'''
print(notes)

<details>
<summary>Click for one reasonable set of answers</summary>

These are *defensible* picks, not the only ones:

1. **Superconducting** or **neutral atom**. Both have many qubits and decent shallow-circuit performance.
2. **Trapped ion**. The lowest gate error rates currently belong to Quantinuum and IonQ.
3. **Neutral atom** (QuEra, Atom Computing). Hundreds-to-thousands of qubits, native programmable connectivity is a fit for QAOA-style combinatorial problems.
4. **Photonic**. Photons travel through fiber natively, so they are the obvious platform for quantum networks.
</details>

---
# Section 5.3: Ethical Implications

Like any powerful technology, quantum AI is going to amplify both good and bad uses. Now is the right time to think about the implications, while the field is still being shaped.

## Responsible AI in the workforce

Three themes come up again and again when researchers discuss responsible quantum AI:

**Transparency.** Quantum algorithms can be even harder to interpret than today's deep neural networks. A trader can lose money on a recommendation from a model that no one can audit. A patient can be denied a drug that a quantum molecular simulation said was unsafe. Building **explainability** into quantum-assisted workflows from the start matters more than retrofitting it later.

**Fairness.** A quantum kernel might find structure in data that humans missed, including spurious correlations that look like patterns. Bias can be amplified, not removed, by sophisticated models. The same fairness audits that good ML teams do now (subgroup performance, counterfactual fairness) apply unchanged to quantum-augmented models.

**Security.** Quantum algorithms (Shor's in particular) can break the public-key cryptography that protects most of today's internet. This is the **harvest-now-decrypt-later** threat: bad actors collecting encrypted data today, planning to decrypt it once large enough quantum computers exist. Migration to **post-quantum cryptography** is happening already; NIST published standards in 2024.

## Workforce impact

The honest picture is mixed.

| Likely effects | What it looks like |
|---|---|
| **New jobs** | Quantum software engineers, quantum-ML practitioners, hardware physicists. Demand outstripping supply right now. |
| **Augmented roles** | Drug-discovery chemists, financial quants, logistics planners gain a faster tool; they don't get replaced by it. |
| **Disrupted roles** | Some specialized optimization or simulation roles may shrink as the toolchain changes. |
| **Skill rebalancing** | Mathematical maturity, ML literacy, and quantum literacy become genuinely useful career skills. |
| **Access inequality** | Quantum hardware is expensive and concentrated in a few companies and countries. That creates winners and losers. |

The historical pattern from previous technology waves (electricity, computers, the internet) suggests **net new job creation** alongside **real local disruption**. Both can be true at once.

## Other ethical angles worth thinking about

- **Environmental:** today's superconducting quantum computers need dilution refrigerators that consume a lot of power. How does that compare to the energy of running the equivalent classical computation?
- **Dual use:** the same algorithms that help drug discovery could be used in the design of bioweapon agents. Same with materials.
- **Hype:** overpromising "quantum advantage" today can produce a Quantum Winter (loss of funding, of trust) if it doesn't deliver. Honest communication matters.

### Activity 5.3: Ethics reflection

Take 10 minutes and write short answers to these prompts. There are no right answers; the goal is to think them through.

1. **Most important.** Which of the responsible-AI themes (transparency, fairness, security) feels most important to you for quantum AI specifically, and why?
2. **Workforce.** Pick a job you know well (your job, a parent's, a friend's). How could quantum AI change it in the next 10 years? Is the change net positive or net negative for the person in that role?
3. **Access.** If only a few companies can afford quantum hardware, what should governments, universities, or non-profits do to keep access fair? Name one concrete policy or program you would back.
4. **Personal line.** Is there a quantum AI application you would refuse to work on if asked? Why?

The cell below is your scratchpad; you will paste short versions into the end-of-unit quiz.

In [ ]:
# ---- Activity 5.3 reflection ----
reflection = {
    "1_most_important_theme":    "...",
    "2_job_impact":              "...",
    "3_access_policy":           "...",
    "4_personal_line":           "...",
}
for k, v in reflection.items():
    print(k.replace("_"," "), ":", v)

---
# Section 5.4: Final Project - Your Industry Proposal

This is the capstone. You will design a **one-page industry proposal**: a short, structured pitch for using quantum AI to solve a real problem in a real industry. Real proposals at companies look exactly like this; the discipline of writing one is a real career skill.

## The proposal template

Fill in each of these fields. Keep each field to a couple of sentences; the whole thing should be readable in under 2 minutes.

1. **Title**: A clear, descriptive name for the project. Example: *"Quantum-enhanced portfolio rebalancing for ESG funds."*
2. **Industry**: Finance, drug discovery, materials, logistics, energy, climate, cybersecurity, agriculture, etc.
3. **Problem**: Describe a *specific* problem in that industry. Not "make medicine better" but "screen 1 million candidate molecules for HER2 binding."
4. **Current approach**: What method is used today? Where does it struggle?
5. **Proposed quantum approach**: Which technique from this course (Grover, QFT, VQE, QAOA, quantum kernel, quantum sampling) would you use, and how would you frame the problem so it fits?
6. **Hardware fit**: Which hardware platform (superconducting, trapped ion, neutral atom, photonic) makes sense given the size of the problem, and why?
7. **Success metric**: How would you measure that your solution worked? "X% faster," "found Y better molecules," "Z reduction in error," etc.
8. **Risks and caveats**: Be honest. What could go wrong? Where might it not deliver?
9. **Ethics check**: Briefly: who benefits, who could be harmed, and what guardrails would you add?

## A worked mini-example

Below is a worked example so you can see the level of detail expected. You will write *your own* in the next cell.

In [ ]:
# A worked example proposal (not yours; just for reference)
example_proposal = {
    "title": "Quantum-assisted catalyst screening for green ammonia production",
    "industry": "Materials discovery and green energy",
    "problem": ("Industrial ammonia synthesis (Haber-Bosch) consumes about 1 to 2 percent "
                "of global energy. Finding a catalyst that lowers the operating temperature "
                "would cut emissions meaningfully."),
    "current_approach": ("DFT (density functional theory) simulation on classical HPC. "
                         "Accurate for small systems but slow and expensive at the "
                         "active-site scale needed."),
    "proposed_quantum_approach": ("Use VQE on a problem-tailored ansatz to estimate the "
                                  "ground-state energy of candidate metal-nitrogen "
                                  "intermediates. Hybrid loop with classical CI to refine."),
    "hardware_fit": ("Trapped-ion hardware (Quantinuum or IonQ): very low gate error matters "
                     "more than qubit count for chemistry accuracy."),
    "success_metric": ("Identify two catalyst candidates whose computed activation barrier "
                       "is at least 20 percent lower than the current Ru/Cs benchmark, "
                       "within 6 months of hardware time."),
    "risks": ("Active site may be too large for current trapped-ion qubit budgets. "
              "Could fall back to embedding the active site in a classical environment."),
    "ethics": ("Net positive: lower emissions. Risk: same catalyst toolkit could help "
               "design environmentally harmful materials too. Mitigation: open publication "
               "of methodology, not of any specific harmful targets."),
}

for k, v in example_proposal.items():
    print(f"--- {k.upper().replace('_',' ')} ---")
    print(v); print()

### Now write your own

Fill in the dictionary below with your proposal. Keep it concise. When you are done, copy the printed output into the matching field of the end-of-unit quiz cell at the bottom of the notebook.

In [ ]:
# ---- Your final-project proposal ----
my_proposal = {
    "title":                        "...",
    "industry":                     "...",
    "problem":                      "...",
    "current_approach":             "...",
    "proposed_quantum_approach":    "...",
    "hardware_fit":                 "...",
    "success_metric":               "...",
    "risks":                        "...",
    "ethics":                       "...",
}

for k, v in my_proposal.items():
    print(f"--- {k.upper().replace('_',' ')} ---")
    print(v); print()

---
# End-of-Unit Quiz

Answer each question. The collapsible blocks reveal a sample answer for self-check.

## Section A: Multiple choice

**1. The acronym NISQ stands for:**

A) Networked Integrated Software-defined Qubits
B) Noisy Intermediate-Scale Quantum
C) Non-Interacting Stochastic Quantum
D) Native Industrial Standard Qubit

<details><summary>Answer 1</summary>**B).** Coined by John Preskill in 2018.</details>

---

**2. Which hardware platform currently has the lowest two-qubit gate error rates?**

A) Superconducting
B) Trapped ion
C) Photonic
D) Topological

<details><summary>Answer 2</summary>**B) Trapped ion.** Quantinuum and IonQ have published two-qubit gate error rates below 1e-3.</details>

---

**3. Why is "harvest-now-decrypt-later" a security concern?**

A) Quantum computers can already decrypt all internet traffic.
B) Encrypted data captured today could be decrypted by a future quantum computer using Shor's algorithm.
C) Quantum random number generators are insecure.
D) Modern encryption is too slow.

<details><summary>Answer 3</summary>**B).** This is why NIST published post-quantum cryptography standards in 2024.</details>

---

**4. A drug company wants to model a small molecule with the highest possible accuracy. The most appropriate near-term hardware is:**

A) The largest possible neutral-atom array.
B) Whichever superconducting device has the most qubits.
C) A trapped-ion device with very low gate error.
D) A photonic chip.

<details><summary>Answer 4</summary>**C).** Accuracy matters more than qubit count for small molecules; trapped ions currently lead on error.</details>

---

**5. Which is the *best* example of a problem that fits QAOA today?**

A) Real-time language translation.
B) Choosing a subset of features for a small classifier.
C) Training a billion-parameter neural net.
D) Predicting tomorrow's stock prices to two decimal places.

<details><summary>Answer 5</summary>**B).** Discrete subset-selection problems fit QAOA naturally; B has a tiny variable count, perfect for current hardware.</details>

## Section B: Short answer (paste your activity answers here)

**6. Case study (Activity 5.1)**
Paste a 3 to 4 sentence summary of your case study: industry, problem, quantum tool you would try, and why.

<details><summary>Sample shape of an answer</summary>"For drug discovery: classical DFT struggles when active sites have more than ~30 strongly correlated electrons. I would try VQE with a hardware-efficient ansatz on a trapped-ion device for accuracy. Success would mean correctly predicting binding energy to within chemical accuracy on a benchmark molecule like FeMoco."</details>

---

**7. Hardware match (Activity 5.2)**
Of the four tasks listed in Activity 5.2, which one did you find hardest to assign to a single platform, and why?

<details><summary>Sample answer</summary>"Task 3 (large optimization) was hardest because neutral-atom hardware has the qubit count but worse error than trapped ion, while trapped ion has the fidelity but fewer qubits. Tradeoff depends on whether the problem tolerates noise."</details>

---

**8. Ethics (Activity 5.3)**
In 2 to 3 sentences, name the responsible-AI theme you find most important for quantum AI (transparency, fairness, or security) and explain why.

<details><summary>Sample answer</summary>Any of the three is defensible. A strong answer ties the choice to a specific scenario, e.g. "Security, because the harvest-now-decrypt-later threat means harm could happen today even before fault-tolerant quantum exists, and migration to post-quantum cryptography needs to happen now."</details>

---

**9. Final project (Activity 5.4)**
Paste your one-page proposal here, or paste your three most important fields (title, problem, proposed quantum approach).

<details><summary>What graders look for</summary>A specific, narrow problem (not "make X better"); a concrete quantum tool with brief justification; an honest hardware choice; a measurable success metric; an honest risk acknowledgment.</details>

---

**10. (Bonus) Big picture.**
In 2 to 3 sentences, summarize what you think the realistic timeline for *meaningful* quantum AI advantage in your chosen industry looks like, and what would have to be true on the hardware side for that to happen.

<details><summary>Sample answer</summary>"For drug discovery: meaningful advantage on lead-compound screening probably 5 to 10 years out, contingent on trapped-ion or neutral-atom platforms reaching ~10,000 logical qubits and error correction overhead dropping. Until then, hybrid VQE on small active sites is the realistic playground."</details>

---

## You're done!

You've completed the final unit. You've now seen the full arc: from foundational concepts in Unit 1, to algorithms in Unit 2, to learning models in Unit 3, to optimization in Unit 4, and finally to real-world impact, hardware limits, ethics, and your own industry proposal in Unit 5.

Run the **End-of-unit submission** cell below to log your completion. Once all 4 graded units (or 5, if your instructor set the bar there) are submitted, your certificate will be emailed to you.

---
## End-of-unit submission

Fill in your quiz answers and any pasted Activity content below, then run this cell to submit. Your answers and the fact that you finished Unit 5 are logged automatically.

In [ ]:
# ============================================================
# END-OF-UNIT SUBMISSION - fill in your answers below
# ============================================================

quiz_answers = {
    "q1": "",        # multiple choice: A, B, C, or D
    "q2": "",
    "q3": "",
    "q4": "",
    "q5": "",
    "q6_case_study":     "Paste your case-study summary here (from Activity 5.1).",
    "q7_hardest_match":  "Which hardware task in 5.2 was hardest to assign and why.",
    "q8_ethics":         "Most important responsible-AI theme and your reasoning.",
    "q9_proposal":       "Your one-page proposal (or the three most important fields).",
    "q10_timeline":      "Realistic timeline + what has to be true on hardware.",
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 5!")
print("This was your final unit. Watch your inbox for a certificate within a minute or so.")